In [12]:
import marimo as mo

In [13]:
# Scorched earth: remove the DB file completely.
# (Not so easy with "real" networked databases.)

import os
FILE = "./objects.db"

if os.path.exists(FILE):
    os.remove(FILE)
    

In [16]:
import requests, io, sqlite3, pandas as pd

# 1. Fetch the data
r = requests.get("https://api.vam.ac.uk/v2/objects/search",
                 {'q': 'India',
                  'page_size': 99,
                  'response_format': 'csv'
                 }
                )
frame = pd.read_csv(io.StringIO(r.content.decode('utf-8')))

# 2. Replace tools.DB with standard sqlite3
# We'll use the file "objects.db" to stay consistent with your earlier cells
conn = sqlite3.connect("objects.db")

# This mimics db.load_from_dataframe
frame.to_sql("objects", conn, if_exists="replace", index=False)

# 3. Create a helper to mimic the db.query() and db.execute() methods
class DBWrapper:
    def __init__(self, connection):
        self.conn = connection
    
    def query(self, sql, params={}):
        # marimo/tools usually uses $var syntax for params; 
        # sqlite3 uses :var or ?
        # This simple wrapper helps pandas read the SQL
        return pd.read_sql_query(sql.replace('$', ':'), self.conn, params=params)
    
    def execute(self, sql, params={}):
        self.conn.execute(sql.replace('$', ':'), params)
        self.conn.commit()

db = DBWrapper(conn)

# Now your original code works as written:
db.query("SELECT * FROM objects")

,accessionNumber,accessionYear,systemNumber,objectType,_primaryTitle,_primaryPlace,_primaryMaker__name,_primaryMaker__association,_primaryDate,_primaryImageId,_sampleMaterial,_sampleTechnique,_sampleStyle,_currentLocation__displayName,_objectContentWarning,_imageContentWarning
0,T.373&A-1974,1974.0,O352221,Pair of shoes,Taj of India,USA,Taj of India,manufacturer,1960s,2015HT7172,NaN,NaN,NaN,In Store,0,0
1,B.75:1-1999,1999.0,O11132,Barbie mystical manipuri,Expressions of India,India,Mattel Toys (India) Ltd,manufacturer,August 1997,2025PK6132,NaN,NaN,NaN,In Store,0,0
2,E.2030-1928,1928.0,O755838,Drawing,Chungqua's premises seen from the portico of t...,Guangzhou,"Chinnery, George",artist,04/03/1828,2025PH5998,paper,drawing (image-making),NaN,"Prints & Drawings Study Room, level C",0,0
3,E.2094-1928,1928.0,O755775,Drawing,The East India Company's 'factory' in Canton (...,Guangzhou,"Chinnery, George",artist,29/09/1825 - 30/05/1852,2025PH6063,paper,drawing (image-making),NaN,"Prints & Drawings Study Room, level C",0,0
4,B.76:1-1999,1999.0,O11139,"Barbie in india, green sari",Barbie in India,India,mattel toys,manufacturer,October 1996,NaN,NaN,NaN,NaN,In Store,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,E.1196-1987,1987.0,O45098,Watercolour,Album of topographical views in India,India,Lady Charlotte Canning,artist,March 1858- November 1858,NaN,watercolour,drawing,NaN,"Prints & Drawings Study Room, level H",0,0
95,E.1195-1987,1987.0,O45097,Watercolour,Album of topographical views in India,India,Lady Charlotte Canning,artist,March 1858- November 1858,2007BM7656,watercolour,drawing,NaN,"Prints & Drawings Study Room, level H",0,0
96,E.1194-1987,1987.0,O45096,Watercolour,Album of topographical views in India,India,Lady Charlotte Canning,artist,March 1858- November 1858,NaN,watercolour,drawing,NaN,"Prints & Drawings Study Room, level H",0,0
97,E.1193-1987,1987.0,O45095,Watercolour,Album of topographical views in India,India,Lady Charlotte Canning,artist,March 1858- November 1858,NaN,watercolour,drawing,NaN,"Prints & Drawings Study Room, level H",0,0


In [17]:
db.execute("""
    CREATE TABLE IF NOT EXISTS locations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        display_name TEXT NOT NULL UNIQUE
    )
""")

db.execute("""
    -- Insert standardized locations
    INSERT INTO locations (display_name)
    SELECT DISTINCT
        CASE
            WHEN lower(_currentLocation__displayName) IN ('in store', 'in storage') THEN 'In Storage'
            ELSE _currentLocation__displayName
        END
    FROM objects
    WHERE _currentLocation__displayName IS NOT NULL
""")

db.execute("""
    -- Add location_id column to main table
    ALTER TABLE objects ADD COLUMN location_id INTEGER REFERENCES locations(id)
""")

db.execute("""
    -- Update the main table with location references
    UPDATE objects
    SET location_id = (
        SELECT id FROM locations
        WHERE display_name =
            CASE
                WHEN lower(objects._currentLocation__displayName) IN ('in store', 'in storage') THEN 'In Storage'
                ELSE objects._currentLocation__displayName
            END
    )
""")

In [18]:
# Examine all locations:

db.query("""
    SELECT * FROM locations
""")

,id,display_name
0,1,In Storage
1,2,"Prints & Drawings Study Room, level C"
2,3,"Prints & Drawings Study Room, level F"
3,4,"Prints & Drawings Study Room, level H"
4,5,"Prints & Drawings Study Room, room 514"


In [19]:
# Select all objects at the "In Storage" location:

db.query("""
    SELECT obj.* FROM objects obj, locations loc
     WHERE loc.id = obj.location_id
       AND loc.display_name = "In Storage"
""")

,accessionNumber,accessionYear,systemNumber,objectType,_primaryTitle,_primaryPlace,_primaryMaker__name,_primaryMaker__association,_primaryDate,_primaryImageId,_sampleMaterial,_sampleTechnique,_sampleStyle,_currentLocation__displayName,_objectContentWarning,_imageContentWarning,location_id
0,T.373&A-1974,1974.0,O352221,Pair of shoes,Taj of India,USA,Taj of India,manufacturer,1960s,2015HT7172,NaN,NaN,NaN,In Store,0,0,1
1,B.75:1-1999,1999.0,O11132,Barbie mystical manipuri,Expressions of India,India,Mattel Toys (India) Ltd,manufacturer,August 1997,2025PK6132,NaN,NaN,NaN,In Store,0,0,1
2,B.76:1-1999,1999.0,O11139,"Barbie in india, green sari",Barbie in India,India,mattel toys,manufacturer,October 1996,NaN,NaN,NaN,NaN,In Store,0,0,1
3,T.504:14-1998,1998.0,O145278,Sample,NaN,India,"Hicks, Sheila",designer,1970,2010ED0258,mounting board,weaving,NaN,In Store,0,0,1
4,T.504:11-1998,1998.0,O145272,Sample,NaN,India,"Hicks, Sheila",designer,1970,2010ED0255,mounting board,weaving,NaN,In Store,0,0,1
5,08101(IS),NaN,O430701,Drawing,Dancing Hall in the Palace of Raja Tirumala Nayak,Madurai,Lt. T. A. Jenkins,painted by,1840 - 1842,2011ER0543,watercolour,drawing (image-making),Company,in storage,0,0,1
6,CIRC.203-1969,1969.0,O96825,Tapestry,Palghat,India,Sheila Hicks,designer,1968,2021NA9607,Cotton,Weaving,NaN,In Store,0,0,1
7,0932(IS),NaN,O72319,Photograph,The Costumes and people of India,India,W. W. Hooper and Surgean G. Western,artists,ca. 1860,2006AW3383,albumen,NaN,NaN,in storage,0,0,1
8,1544-1877,1877.0,O169939,Medal,NaN,England,"Wyon, William",artist,1848,NaN,silver,NaN,NaN,in storage,0,0,1
9,IS.84-1887,1887.0,O119044,Oil painting,Copy of painting inside the caves of Ajanta (c...,Ajanta,John Griffiths,artist,1881-1883,2006AY0554,oil colour,oil painting,NaN,In Store,0,0,1


In [22]:
from flask import Flask, jsonify
from flask_cors import CORS, cross_origin

_app = Flask(__name__)
# This is the "Magic Key" that fixes the Access-Control-Allow-Origin error
CORS(_app) 

@_app.route("/locations")
def _get_locations():
    try:
        # Ensure 'db' was defined in the previous cell!
        locations_frame = db.query("SELECT id, display_name FROM locations")
        records = locations_frame.to_records(index=False)
        return { "data" : [{"id" : int(r['id']), "name" : r['display_name']} for r in records] }
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@_app.route("/objects_at_location/<loc_id>")
def _get_objects(loc_id):
    try:
        locations_frame = db.query("SELECT _primaryTitle FROM objects WHERE location_id = :loc_id", {"loc_id" : loc_id})
        records = locations_frame.to_records(index=False)
        return { "data" : [{"title" : r['_primaryTitle']} for r in records] }
    except Exception as e:
        return jsonify({"error": str(e)}), 500

_app.run(port=5051) # Using 5051 since 5050 was blocked

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5051
Press CTRL+C to quit
127.0.0.1 - - [20/Apr/2026 14:25:54] "GET /locations HTTP/1.1" 500 -
127.0.0.1 - - [20/Apr/2026 14:25:55] "GET /locations HTTP/1.1" 500 -
127.0.0.1 - - [20/Apr/2026 14:25:55] "GET /locations HTTP/1.1" 500 -
127.0.0.1 - - [20/Apr/2026 14:25:56] "GET /locations HTTP/1.1" 500 -
